#Spark Basics Assignment

import packages and read df

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, sum as _sum, min, max, mean, rand, when, floor

spark = SparkSession.builder.appName("sparky").getOrCreate()

df = spark.read.csv("../data/superstore.csv",header=True,inferSchema=True,multiLine=True,escape='"')

# Change datatypes from string to numerical
df = (df
    .withColumn("Sales", col("Sales").cast("double"))
    .withColumn("Quantity", col("Quantity").cast("int"))
    .withColumn("Discount", col("Discount").cast("double")))

# New columns to solve all questions in assg
df = (df
    .withColumn("Age", (floor(rand() * 50) + 18).cast("int"))
    .withColumn("Subscription", when(rand() > 0.5, "Premium").otherwise("Standard"))
    .withColumn("Email", col("Customer Name"))
    .withColumn("Username", col("Customer ID")))

df.printSchema()
df.show(5)

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: date (nullable = true)
 |-- Ship Date: date (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Subscription: string (nullable = false)
 |-- Email: string (nullable = true)
 |-- Username: string (nullable

Q1: What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing?

A: In tradition MapReduce it splits tasks and performs them one by one but while doing so, it reads from disk, performs a task then writes to disk, this keeps looping. RAM is faster than SSD so more time is wasted in read write rather than doing the actual computations. In spark we get easy to use api's and spark reads, stores in RAM, performs all needed tasks and then writes. So read write only happens once which saves time.

Q2: Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.

A: In disk-based systems, in ML algos the same dataset is used again and again while the predicted value gets better and better till convergence but every iteration has to read this dataset, what happens is that if the dataset is huge then the read is slow, and 100 iterations means 100 slow reads. Instead of doing that spark allows df.cache() which stores the dataset in RAM till all iterations are completed.

Q3: Write a code snippet to remove all duplicate rows from a DataFrame based on a specific set of columns: user_id and transaction_date. 

In [2]:
df_q3 = df.dropDuplicates(["Customer ID", "Order Date"])
df_q3.select("Customer ID", "Order Date").show()

+-----------+----------+
|Customer ID|Order Date|
+-----------+----------+
|   JH-15910|2016-10-28|
|   EB-13705|2017-11-12|
|   TW-21025|2017-07-21|
|   EA-14035|2016-02-16|
|   GM-14500|2017-08-20|
|   DR-12940|2015-12-27|
|   VG-21790|2017-07-08|
|   MO-17500|2017-10-19|
|   TZ-21580|2015-12-04|
|   EH-14185|2017-09-16|
|   CP-12340|2015-11-07|
|   CG-12040|2016-10-24|
|   RB-19795|2014-06-03|
|   RB-19465|2017-10-20|
|   JD-16015|2014-10-06|
|   GZ-14470|2015-11-14|
|   JF-15415|2015-10-25|
|   ME-17320|2014-01-06|
|   SP-20620|2017-11-13|
|   DK-12835|2016-03-25|
+-----------+----------+
only showing top 20 rows


Q4: Given a DataFrame df_sales, write a query to filter for rows where the region is 'West' and then group by product_category to find the average sale_amount.

In [3]:
df_q4 = (df.filter(col("Region") == "West")
    .groupBy("Category")
    .agg(avg("Sales").alias("avg_sales")))
df_q4.show()

+---------------+----------------+
|       Category|       avg_sales|
+---------------+----------------+
|Office Supplies|116.422376910912|
|      Furniture|357.302324611033|
|     Technology|420.687532554257|
+---------------+----------------+



Q5: What is the difference between .na.drop() and .na.fill()? Provide a code example of filling null values in a status column with the string 'Unknown'.

A: drop() removes row, fill() replaces NULL value with the param val

In [4]:
df_q5 = df.na.fill({"Ship Mode": "Unknown"})

Q6: Write a query to find the total count of records for each city in a DataFrame, but only for cities where the count is greater than 100. 

In [5]:
df_q6 = (df.groupBy("City")
    .count()
    .filter(col("count") > 100))
df_q6.show()

+-------------+-----+
|         City|count|
+-------------+-----+
|  Springfield|  163|
|       Dallas|  157|
| Philadelphia|  537|
|  Los Angeles|  747|
|San Francisco|  510|
|    San Diego|  170|
|      Detroit|  115|
|     Columbus|  222|
|      Chicago|  314|
|      Seattle|  428|
|New York City|  915|
|      Houston|  377|
| Jacksonville|  125|
+-------------+-----+



Q7: How does the immutability of Spark DataFrames affect how you perform "data cleaning" steps like dropping columns or renaming them? 

A: In spark the df is immutable which means it cannot be edited, spark just creates a new df for the change so df.drop() does nothing unless its df = df.drop() in which the edited copy replaces the original. This is why we chain functions in spark. making new dataframes helps in preventing inconsistencies.

Q8: Write a Spark command to filter a dataset for rows where the age is between 18 and 30 (inclusive) and the subscription is 'Premium'. 

In [6]:
df_q8 = df.filter((col("Age").between(18, 30)) & (col("Subscription") == "Premium"))
df_q8.select("Customer Name","Age").show()

+-----------------+---+
|    Customer Name|Age|
+-----------------+---+
|    Harold Pawlan| 19|
|  Alejandro Grove| 21|
|  Tracy Blumstein| 22|
|  Tracy Blumstein| 21|
|  Tracy Blumstein| 30|
|  Ted Butterfield| 23|
|       Joel Eaton| 24|
|   Parhena Norris| 21|
|    Lena Cacioppo| 22|
|    Lena Cacioppo| 29|
|      Clay Ludtke| 29|
|Steven Cartwright| 30|
| Jonathan Doherty| 28|
|      Dave Brooks| 25|
|         Jim Kriz| 18|
|   David Kendrick| 30|
|      Mark Packer| 24|
|      Joseph Holt| 29|
|  Victoria Wilson| 29|
|   Joni Blumstein| 25|
+-----------------+---+
only showing top 20 rows


Q9: When cleaning a dataset, why is it often better to handle null values before performing mathematical aggregations like sum() or avg()? 

A: Its better because spark assumes NULL as missing rather than 0 and just skips it, it ignores the row in calculation. so even if half the data is not there, it still works but the answer isn't incorrect but its not what you wanted, so either drop those or fill nulls.

Q10: Write the code to revise a column named raw_timestamp by casting it to a TimestampType and renaming it to event_time. 

In [7]:
df_q10 = df.withColumn("event_time", col("Order Date").cast("timestamp")).drop("Order Date")
df_q10.select("Order ID", "event_time").show()

+--------------+-------------------+
|      Order ID|         event_time|
+--------------+-------------------+
|CA-2016-152156|2016-11-08 00:00:00|
|CA-2016-152156|2016-11-08 00:00:00|
|CA-2016-138688|2016-06-12 00:00:00|
|US-2015-108966|2015-10-11 00:00:00|
|US-2015-108966|2015-10-11 00:00:00|
|CA-2014-115812|2014-06-09 00:00:00|
|CA-2014-115812|2014-06-09 00:00:00|
|CA-2014-115812|2014-06-09 00:00:00|
|CA-2014-115812|2014-06-09 00:00:00|
|CA-2014-115812|2014-06-09 00:00:00|
|CA-2014-115812|2014-06-09 00:00:00|
|CA-2014-115812|2014-06-09 00:00:00|
|CA-2017-114412|2017-04-15 00:00:00|
|CA-2016-161389|2016-12-05 00:00:00|
|US-2015-118983|2015-11-22 00:00:00|
|US-2015-118983|2015-11-22 00:00:00|
|CA-2014-105893|2014-11-11 00:00:00|
|CA-2014-167164|2014-05-13 00:00:00|
|CA-2014-143336|2014-08-27 00:00:00|
|CA-2014-143336|2014-08-27 00:00:00|
+--------------+-------------------+
only showing top 20 rows


Q11: Explain the "Shuffle" process that occurs during a grouping operation. Why is it considered a wide transformation? 

A: shuffle is very expensive, for groupbys spark needs the same key rows to be together but in reality they can be in different partitions, spark has to physically move them to the same partition, this network transfer is slower than in-memory. Its considered wide because output partition depends on multiple input partitions.

Q12: Write a code snippet that identifies and removes rows where the email column contains null values OR the username is an empty string. 


In [8]:
df_q12 = df.filter(col("Email").isNotNull() & (col("Username") != ""))

Q13: How do you use the .agg() function to calculate multiple statistics at once, such as the min, max, and mean of the price column? 


In [9]:
df.agg(
    min("Sales").alias("min_sales"),
    max("Sales").alias("max_sales"),
    mean("Sales").alias("mean_sales")
).show()

+---------+---------+-----------------+
|min_sales|max_sales|       mean_sales|
+---------+---------+-----------------+
|    0.444| 22638.48|229.8580008304938|
+---------+---------+-----------------+



Q14: In the context of cleaning a dataset, what is the risk of using inferSchema=true when your source data contains messy or inconsistent date formats?

A: It does happen in this dataset, inferschema makes spark guess the datatype, if spark reads values in col and some of it are unknown, they consider the datatype as string becuase atleast one value is not numeric. dates turn to NULLs if mismatch. The bad thing is that the pipeline succeeds but its bad data, thats why structtype is set to be caught if mismatched. Here embedded quoted values caused a bad read which put a string in a numerical column and double cast got busted.

Q15: Write a final processing pipeline that: 
Filters out duplicates. 
Fills null prices with 0. 
Groups by store_id to calculate total revenue. 

In [10]:
pipeline_result = (df
    .dropDuplicates()
    .na.fill({"Sales": 0})
    .groupBy("State")
    .agg(_sum("Sales").alias("total_revenue")))
pipeline_result.show()

+--------------------+------------------+
|               State|     total_revenue|
+--------------------+------------------+
|                Utah|11220.055999999999|
|           Minnesota|29863.150000000005|
|                Ohio| 78258.13599999997|
|              Oregon|          17431.15|
|            Arkansas|11678.130000000001|
|               Texas|170188.04579999985|
|        North Dakota| 919.9099999999999|
|        Pennsylvania|116511.91400000002|
|         Connecticut|13384.356999999995|
|             Vermont| 8929.369999999999|
|            Nebraska|           7464.93|
|              Nevada|         16729.102|
|          Washington|138641.26999999996|
|            Illinois|         80166.101|
|            Oklahoma|19683.390000000003|
|District of Columbia|2865.0200000000004|
|            Delaware|27451.068999999996|
|          New Mexico|          4783.522|
|       West Virginia|          1209.824|
|            Missouri|          22205.15|
+--------------------+------------

There was no store_id, so used state, same logic